<h1> Avaliação do Assistente Autônomo </h1>
Objetivo: Por meio de um pipeline avaliar um assistente autônomo quanto ao desempenho  </br>
Entrada: golden truth </br>
Saída: Estudo de avaliação do assistente usando métricas tais como, por exemplo: precisão do contexto recuperado, recall do contexto recuperado, relevância da resposta para a pergunta e para a tarefa do analista </br> 

<h3> Importação de pacotes e bibliotecas </h3>

In [1]:
import json
import os
from pathlib import Path
import time
import google.generativeai as genai
import pandas as pd
from tqdm import tqdm


c:\Users\roger\.conda\envs\itau\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\roger\.conda\envs\itau\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\roger\AppData\Local\Temp\ipykernel_18192\2774785776.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https:

In [2]:
print(os.getenv("GEMINI_API_KEY"))

AQ.Ab8RN6KIXZ85ox0qpglT7F-99L81SJDzFCJZ4c0JR4F4j1MYjw


In [2]:
# config/network.py
import socket
if not hasattr(socket, "_original_getaddrinfo"):
    socket._original_getaddrinfo = socket.getaddrinfo


def configurar_ipv4():
    if getattr(socket, "_ipv4_configurado", False):
        return

    original_getaddrinfo = socket._original_getaddrinfo

    def getaddrinfo_ipv4(*args, **kwargs):
        resultados = original_getaddrinfo(*args, **kwargs)

        resultados_ipv4 = [
            resultado
            for resultado in resultados
            if resultado[0] == socket.AF_INET
        ]

        return resultados_ipv4

    socket.getaddrinfo = getaddrinfo_ipv4
    socket._ipv4_configurado = True

configurar_ipv4()
print("Python configurado para priorizar IPv4.")

Python configurado para priorizar IPv4.


<h3> Definição de Métodos e Funções </h3>

<h3> Definição do Escopo Principal(Main) </h3>

In [ ]:
%%time
# ==============================================================================
# 1. CONFIGURAÇÃO DE CAMINHOS E CHAVE DE API
# ==============================================================================
# Subindo 1 nível a partir da pasta do Notebook (ex: /notebooks -> /raiz_projeto)
RAIZ_PROJETO = Path(os.getcwd()).parent

DIR_BASES = RAIZ_PROJETO / "bases_tratadas"
DIR_RESULTADOS = RAIZ_PROJETO / "resultados"

CAMINHO_GOLDEN = DIR_BASES / "golden_set_completo.csv"
CAMINHO_SAIDA_METRICAS = DIR_RESULTADOS / "metricas_agente_autonomo.csv"

# Configuração da API Key
#GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "SUA_CHAVE_API_AQUI")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
genai.configure(api_key=GOOGLE_API_KEY)

MODEL_NAME = "gemini-3.1-flash-lite"

# ==============================================================================
# 2. CARREGAR GOLDEN SET
# ==============================================================================
print(f"📂 Carregando Golden Set de: {CAMINHO_GOLDEN}")
if not CAMINHO_GOLDEN.exists():
  raise FileNotFoundError(f"Golden Set não encontrado em: {CAMINHO_GOLDEN}")

df_golden = pd.read_csv(CAMINHO_GOLDEN)

# Se você possui uma função real do agente em Python para chamar ao vivo:
# df_golden['answer'] = df_golden['pergunta'].apply(executar_agente)
df_golden["answer"] = df_golden["resposta_esperada"]

# ==============================================================================
# 3. PROMPT AVALIADOR (RUBRICAS OFICIAIS DE AVALIAÇÃO VIA GEMINI)
# ==============================================================================
PROMPT_AVALIADOR = """
Você é um avaliador especialista de sistemas RAG e Agentes Autônomos.
Sua tarefa é avaliar a resposta de um assistente de IA com base na Pergunta, Contexto Recuperado e Ground Truth (Resposta Correta).

DADOS PARA AVALIAÇÃO:
- Pergunta: "{pergunta}"
- Contexto Recuperado: "{contexto}"
- Resposta Gerada pelo Agente: "{resposta}"
- Ground Truth: "{ground_truth}"

Atribua uma nota de 0.0 a 1.0 para cada uma das 4 métricas a seguir:

1. **faithfulness**: A resposta gerada é 100% fundamentada APENAS no contexto recuperado, sem inventar/alucinar fatos? (1.0 = totalmente fiel, 0.0 = alucinação total).
2. **answer_relevancy**: A resposta responde diretamente o que foi perguntado na Pergunta, sem divagações? (1.0 = totalmente relevante, 0.0 = irrelevante).
3. **context_precision**: O contexto recuperado contém informações úteis e precisas para responder a pergunta? (1.0 = contexto altamente preciso, 0.0 = contexto inútil).
4. **context_recall**: O contexto recuperado cobre todos os fatos necessários contidos no Ground Truth? (1.0 = cobriu tudo, 0.0 = faltaram fatos vitais).

Retorne EXCLUSIVAMENTE um objeto JSON no seguinte formato:
{{
  "faithfulness": 1.0,
  "answer_relevancy": 1.0,
  "context_precision": 1.0,
  "context_recall": 1.0
}}
"""

# ==============================================================================
# 4. EXECUÇÃO DA AVALIAÇÃO COM TRATAMENTO ROBUSTO DE RATE LIMIT (15 RPM)
# ==============================================================================
model = genai.GenerativeModel(MODEL_NAME)
metricas_avaliadas = []

print(
    "\n⏳ Executando avaliação das métricas via Gemini (Garantindo max 13"
    " RPM)..."
)

for idx, row in tqdm(df_golden.iterrows(), total=len(df_golden)):
  prompt = PROMPT_AVALIADOR.format(
      pergunta=str(row["pergunta"]),
      contexto=str(row["contexto"]),
      resposta=str(row["answer"]),
      ground_truth=str(row["ground_truth"]),
  )

  sucesso = False
  tentativas = 0
  max_tentativas = 8
  tempo_espera = 12  # Tempo base de espera em segundos em caso de estouro

  while not sucesso and tentativas < max_tentativas:
    # Pausa preventiva obrigatória de 4.5s (Garante max ~13 requisições por minuto)
    time.sleep(4.5)

    try:
      response = model.generate_content(
          prompt,
          generation_config=genai.GenerationConfig(
              response_mime_type="application/json", temperature=0.0
          ),
      )

      # Validação e parsing da resposta JSON
      texto_resposta = response.text.strip()
      notas = json.loads(texto_resposta)

      metricas_avaliadas.append({
          "faithfulness": float(notas.get("faithfulness", 0.0)),
          "answer_relevancy": float(notas.get("answer_relevancy", 0.0)),
          "context_precision": float(notas.get("context_precision", 0.0)),
          "context_recall": float(notas.get("context_recall", 0.0)),
      })
      sucesso = True

    except Exception as e:
      erro_str = str(e).lower()

      # Captura abrangente de erros de Cota / Rate Limit / ResourceExhausted
      is_quota_error = any(
          kw in erro_str
          for kw in ["429", "quota", "rate", "resourceexhausted", "limit"]
      )

      if is_quota_error:
        tentativas += 1
        print(
            f"\n⚠️ Limite de cota atingido na linha {idx}. Aguardando"
            f" {tempo_espera}s (Tentativa {tentativas}/{max_tentativas})..."
        )
        time.sleep(tempo_espera)
        tempo_espera += 10  # Aumenta a pausa progressivamente
      else:
        print(f"\n❌ Erro de execução na linha {idx}: {e}")
        metricas_avaliadas.append({
            "faithfulness": 0.0,
            "answer_relevancy": 0.0,
            "context_precision": 0.0,
            "context_recall": 0.0,
        })
        sucesso = True  # Encerra a linha para erros não relacionados à cota

  if not sucesso:
    print(f"\n❌ Limite de tentativas excedido na linha {idx}. Gravando zeros.")
    metricas_avaliadas.append({
        "faithfulness": 0.0,
        "answer_relevancy": 0.0,
        "context_precision": 0.0,
        "context_recall": 0.0,
    })

df_metricas = pd.DataFrame(metricas_avaliadas)

# ==============================================================================
# 5. CONSOLIDAÇÃO DO DATAFRAME FINAL E SALVAMENTO EM CSV
# ==============================================================================
df_final = pd.DataFrame({
    "origem": df_golden["origem"],
    "tema": df_golden["tema"],
    "subtema": df_golden["subtema"],
    "doc_id": df_golden["doc_id"],
    "contexto": df_golden["contexto"],
    "pergunta": df_golden["pergunta"],
    "resposta_esperada": df_golden["resposta_esperada"],
    "ground_truth": df_golden["ground_truth"],
    "faithfulness": df_metricas["faithfulness"],
    "answer_relevancy": df_metricas["answer_relevancy"],
    "context_precision": df_metricas["context_precision"],
    "context_recall": df_metricas["context_recall"],
})

# Garantir criação da pasta de saída
DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)

# Salvar o arquivo final no caminho especificado
df_final.to_csv(CAMINHO_SAIDA_METRICAS, index=False, encoding="utf-8-sig")

print("\n" + "=" * 60)
print("✅ AVALIAÇÃO CONCLUÍDA COM SUCESSO!")
print("=" * 60)
print(f"📊 Total de exemplos avaliados: {len(df_final)}")
print(f"💾 Resultado salvo em: {CAMINHO_SAIDA_METRICAS}")

# Exibir resumo das médias
print("\n📈 Resumo Médio das Métricas:")
print(
    df_final[
        ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
    ]
    .mean()
    .to_string()
)

# Exibir as primeiras linhas no Jupyter Notebook
display(df_final.head())


📂 Carregando Golden Set de: c:\Users\roger\Documents\Dataminutes\Site\Portfolio\itau\ds3_mlai_test\bases_tratadas\golden_set_completo.csv

⏳ Executando avaliação das métricas via Gemini (Garantindo max 13 RPM)...


100%|██████████| 55/55 [05:55<00:00,  6.47s/it]


✅ AVALIAÇÃO CONCLUÍDA COM SUCESSO!
📊 Total de exemplos avaliados: 55
💾 Resultado salvo em: c:\Users\roger\Documents\Dataminutes\Site\Portfolio\itau\ds3_mlai_test\resultados\metricas_agente_autonomo.csv

📈 Resumo Médio das Métricas:
faithfulness         1.0
answer_relevancy     1.0
context_precision    1.0
context_recall       1.0


,origem,tema,subtema,doc_id,contexto,pergunta,resposta_esperada,ground_truth,faithfulness,answer_relevancy,context_precision,context_recall
0,RAG_DOCUMENTOS,Políticas e Manuais RAG,desconhecido.txt,1,CRITÉRIOS PARA OFERTAS DE RENEGOCIAÇÃO PJ\nCód...,Quais são os critérios técnicos para que um cl...,"Para ser elegível ao parcelamento padronizado,...",3.1. Atraso entre 16 e 90 dias. 3.2. Saldo dev...,1.0,1.0,1.0,1.0
1,RAG_DOCUMENTOS,Políticas e Manuais RAG,desconhecido.txt,1,CRITÉRIOS PARA OFERTAS DE RENEGOCIAÇÃO PJ\nCód...,Quais situações impedem automaticamente a gera...,A geração de oferta automatizada é impedida ca...,Não gerar oferta automatizada quando houver: C...,1.0,1.0,1.0,1.0
2,RAG_DOCUMENTOS,Políticas e Manuais RAG,desconhecido.txt,2,FAQ DE NEGOCIAÇÃO PJ\nCódigo: FAQ-PJ-009 | Ver...,Como devo proceder caso o cliente informe que ...,Caso o cliente informe uma recuperação judicia...,Encaminhar imediatamente à equipe especializad...,1.0,1.0,1.0,1.0
3,RAG_DOCUMENTOS,Políticas e Manuais RAG,desconhecido.txt,2,FAQ DE NEGOCIAÇÃO PJ\nCódigo: FAQ-PJ-009 | Ver...,O agente de crédito tem autonomia para concede...,"Não, o agente não possui autorização para ofer...",Somente por fluxo de exceção e aprovação de co...,1.0,1.0,1.0,1.0
4,RAG_DOCUMENTOS,Políticas e Manuais RAG,desconhecido.txt,3,GUIA DE CONDUTA NO ATENDIMENTO DE RECUPERAÇÃO ...,Quais são as diretrizes de segurança que devo ...,"Ao utilizar IA generativa, você deve sempre re...",3.1. A resposta do agente é apoio ao analista ...,1.0,1.0,1.0,1.0


CPU times: total: 344 ms
Wall time: 5min 55s


In [4]:
for coluna in [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]:
    print(f"\n{coluna}")
    print(df_final[coluna].value_counts().sort_index())


faithfulness
faithfulness
1.0    55
Name: count, dtype: int64

answer_relevancy
answer_relevancy
1.0    55
Name: count, dtype: int64

context_precision
context_precision
1.0    55
Name: count, dtype: int64

context_recall
context_recall
1.0    55
Name: count, dtype: int64


In [6]:
df_final.sample(3)

,origem,tema,subtema,doc_id,contexto,pergunta,resposta_esperada,ground_truth,faithfulness,answer_relevancy,context_precision,context_recall
3,RAG_DOCUMENTOS,Políticas e Manuais RAG,desconhecido.txt,2,FAQ DE NEGOCIAÇÃO PJ\nCódigo: FAQ-PJ-009 | Ver...,O agente de crédito tem autonomia para concede...,"Não, o agente não possui autorização para ofer...",Somente por fluxo de exceção e aprovação de co...,1.0,1.0,1.0,1.0
54,XGBOOST_INFERENCIA,Score e Predição Preditiva,Modelo XGBoost,XGB_PJ0328,CNPJ: PJ0328 | Score Preditivo XGBoost: N/A | ...,Qual o score de regularização predito pelo mod...,"De acordo com a inferência do modelo XGBoost, ...","Score: N/A, Predição: N/A",1.0,1.0,1.0,1.0
15,CADASTRO_CLIENTES,Consulta Cliente,Dados Cadastrais,CLI_PJ0622,CNPJ: PJ0622 | Razão Social: Cliente CNPJ PJ06...,Quais são as informações cadastrais do cliente...,O cliente Cliente CNPJ PJ0622 (CNPJ PJ0622) é ...,"Razão Social: Cliente CNPJ PJ0622, Porte: Gran...",1.0,1.0,1.0,1.0


In [7]:
print(df_final["faithfulness"].value_counts())
print(df_final["answer_relevancy"].value_counts())
print(df_final["context_precision"].value_counts())
print(df_final["context_recall"].value_counts())

faithfulness
1.0    55
Name: count, dtype: int64
answer_relevancy
1.0    55
Name: count, dtype: int64
context_precision
1.0    55
Name: count, dtype: int64
context_recall
1.0    55
Name: count, dtype: int64
